In [62]:
import pandas as pd

In [63]:
orders = pd.read_csv("../data/processed/orders_clean.csv")
products = pd.read_csv("../data/processed/products_clean.csv")
order_reviews = pd.read_csv("../data/processed/order_reviews_clean.csv")
customers = pd.read_csv("../data/processed/customers_clean.csv")
order_items = pd.read_csv("../data/processed/order_items_clean.csv")
order_payments = pd.read_csv("../data/processed/order_payments_clean.csv")
sellers = pd.read_csv("../data/processed/sellers_clean.csv")
geo_locations = pd.read_csv("../data/processed/geolocation_clean.csv")
product_category_name_translations = pd.read_csv("../data/processed/category_translation_clean.csv")
closed_deals = pd.read_csv("../data/processed/closed_deals_clean.csv")
marketing_leads = pd.read_csv("../data/processed/marketing_leads_clean.csv")

In [64]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

I'm creating columns to allow me to perform analyses based on specific periods, such as years or months.

In [65]:
orders = orders.assign(
    purchase_year=orders["order_purchase_timestamp"].dt.year,
    purchase_month=orders["order_purchase_timestamp"].dt.month,
    purchase_day=orders["order_purchase_timestamp"].dt.day,
    purchase_hour=orders["order_purchase_timestamp"].dt.hour,
    purchase_weekday=orders["order_purchase_timestamp"].dt.day_name()
)

These features enable temporal analyses such as monthly sales trends, hourly purchasing behavior, and weekday purchasing patterns.

In [66]:
orders[
    [
        "purchase_month",
        "purchase_day",
        "purchase_hour",
        "purchase_weekday",
        "purchase_year"
    ]
].head()

,purchase_month,purchase_day,purchase_hour,purchase_weekday,purchase_year
0,10,2,10,Monday,2017
1,7,24,20,Tuesday,2018
2,8,8,8,Wednesday,2018
3,11,18,19,Saturday,2017
4,2,13,21,Tuesday,2018


I implement validation practices like this. Instead of just saying "the code ran," I examine whether the generated values ​​make sense.

In [67]:
orders["purchase_hour"].describe()

count    99441.000000
mean        14.770829
std          5.326800
min          0.000000
25%         11.000000
50%         15.000000
75%         19.000000
max         23.000000
Name: purchase_hour, dtype: float64

In [68]:
orders["purchase_month"].value_counts().sort_index()

purchase_month
1      8069
2      8508
3      9893
4      9343
5     10573
6      9412
7     10318
8     10843
9      4305
10     4959
11     7544
12     5674
Name: count, dtype: int64

I'm developing a feature to differentiate between weekday and weekend order statuses.

In [69]:
orders["is_weekend"] = orders["purchase_weekday"].isin(
    ["Saturday", "Sunday"]
)

In [70]:
orders["is_weekend"].value_counts()

is_weekend
False    76594
True     22847
Name: count, dtype: int64

In [71]:
orders["is_weekend"].value_counts(normalize=True)

is_weekend
False    0.770246
True     0.229754
Name: proportion, dtype: float64

I'm investigating the question, "How many hours did it take for the order to be confirmed?" The relationship between order confirmation efficiency, delivery performance, and customer satisfaction can be examined.

In [72]:
orders["approval_time_hours"] = (
    orders["order_approved_at"] -
    orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

The distribution appears to be skewed to the right. This means that the vast majority of orders were confirmed quickly, but orders confirmed very late have pushed the average upwards. The order confirmed in 4509 hours may have been recorded late in the system, there may have been a data entry problem, a data error, or the payment may have been confirmed months later.

In [73]:
orders["approval_time_hours"].describe()

count    99281.000000
mean        10.419094
std         26.038004
min          0.000000
25%          0.215000
50%          0.343333
75%         14.580833
max       4509.180556
Name: approval_time_hours, dtype: float64

I left 160 NaN in the order_approved_at column during the cleaning phase. Therefore, it's normal to see this value here as well.;

In [74]:
orders["approval_time_hours"].isna().sum()

np.int64(160)

In [75]:
orders.loc[
    orders["approval_time_hours"] < 0,
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "approval_time_hours"
    ]
]

,order_purchase_timestamp,order_approved_at,approval_time_hours


I'm creating a shipping_time feature to analyze the distribution of orders for shipment/order preparation.

In [76]:
orders["shipping_time_days"] = (
    orders["order_delivered_carrier_date"]
    - orders["order_approved_at"]
).dt.total_seconds() / (60 * 60 * 24)

Some orders appear to have been shipped before confirmation. This is contrary to normal workflow. 125 days for shipping is also a long time. There may have been a stock issue, the system may have updated late, or the order may have been delayed.

In [77]:
orders["shipping_time_days"].describe()

count    97644.000000
mean         2.805038
std          3.549427
min       -171.219005
25%          0.875509
50%          1.818397
75%          3.580469
max        125.762569
Name: shipping_time_days, dtype: float64

I am reviewing orders with incorrect delivery times. Some confirmation times (such as 23:31) are duplicated. In this case, the confirmation time may not be the actual confirmation time; it may have been assigned or rounded off by the system later.

In [78]:
orders.loc[
    orders["shipping_time_days"] < 0,
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "shipping_time_days"
    ]
]

,order_approved_at,order_delivered_carrier_date,shipping_time_days
15,2018-06-12 23:31:02,2018-06-11 14:54:00,-1.359051
64,2018-04-24 18:25:22,2018-04-23 19:19:14,-0.962593
199,2018-07-26 23:31:53,2018-07-24 12:57:00,-2.440891
210,2018-07-23 12:31:53,2018-07-23 12:24:00,-0.005475
415,2018-07-27 23:31:09,2018-07-24 14:03:00,-3.394549
...,...,...,...
99091,2018-07-05 16:17:59,2018-07-05 14:11:00,-0.088183
99230,2018-07-05 16:32:52,2018-07-03 12:57:00,-2.149907
99266,2018-02-04 23:31:46,2018-01-31 18:11:58,-4.222083
99377,2018-04-24 19:26:10,2018-04-23 17:18:40,-1.088542


A subset of orders shows negative shipping times because the recorded carrier pickup timestamp precedes the recorded approval timestamp. Since the approval times of these orders are also unusually long, this likely indicates timestamp inconsistencies or data quality issues rather than actual business events. The records are retained for transparency and will be considered during later analyses. This is derived data quality issue.

In [79]:
orders.loc[
    orders["shipping_time_days"] < 0,
    "approval_time_hours"
].describe()

count    1359.000000
mean       54.519698
std        49.042012
min         0.124444
25%        20.629722
50%        46.232778
75%        78.489306
max       291.410556
Name: approval_time_hours, dtype: float64

In [80]:
orders.loc[
    orders["shipping_time_days"] < 0,
    "order_status"
].value_counts()

order_status
delivered    1350
shipped         9
Name: count, dtype: int64

1797 NaNs are normal. There were 160 NaN in the order_approved_at column and 1783 NaN in the order_delivered_carrier_date column. Therefore, it is normal to have a NaN value that is more than 1783, but less than the sum of these two numbers.

In [81]:
orders["shipping_time_days"].isna().sum()


np.int64(1797)

I'm examining the delivery time values, which are one of the most important features in the orders table. This feature measures the customer's end-to-end delivery experience and supports analytics related to delivery performance, customer satisfaction, and operational efficiency.

In [82]:
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

In [83]:
orders["delivery_time_days"].describe()

count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
Name: delivery_time_days, dtype: float64

These NaN values ​​are normal because there were orders where we left NaN in the order_delivered_customer_date column during the data cleanup process.

In [84]:
orders["delivery_time_days"].isna().sum()

np.int64(2965)

In [85]:
orders.loc[
    orders["delivery_time_days"] < 0,
    [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_time_days"
    ]
]

,order_purchase_timestamp,order_delivered_customer_date,delivery_time_days


So far, I've examined time-related properties individually in separate code snippets for illustrative purposes. However, these calculations can be done much faster using functions.

I look at the difference between the estimated delivery date the business tells the customer and the actual delivery date. This allows me to mark some orders as "late delivery".

In [86]:
orders["estimated_delivery_gap_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

Overall, the delivery was made earlier than expected. The -146 and +188 values ​​here are likely outliers. They will be investigated further.

In [87]:
orders["estimated_delivery_gap_days"].describe()

count    96476.000000
mean       -11.179120
std         10.186113
min       -146.016123
25%        -16.244384
50%        -11.948941
75%         -6.390000
max        188.975081
Name: estimated_delivery_gap_days, dtype: float64

In [88]:
orders["estimated_delivery_gap_days"].isna().sum()

np.int64(2965)

In [89]:
orders["estimated_delivery_gap_days"].value_counts().head(10)

estimated_delivery_gap_days
-12.385602    5
-15.084213    4
-7.181331     4
-13.144051    4
-13.283183    4
-8.185579     4
-7.216123     4
-13.286227    4
-9.091238     4
-9.213519     4
Name: count, dtype: int64

Based on this difference, we will derive an "is_late_delivery" feature for late delivery.

In [93]:
orders["is_late_delivery"] = (
    orders["estimated_delivery_gap_days"] > 0
)
orders["is_late_delivery"]

0        False
1        False
2        False
3        False
4        False
         ...  
99436    False
99437    False
99438    False
99439    False
99440    False
Name: is_late_delivery, Length: 99441, dtype: bool

I'm looking at the number of orders that were delivered late.

In [ ]:
orders["is_late_delivery"].value_counts()

is_late_delivery
False    91614
True      7827
Name: count, dtype: int64

A new categorical feature, `purchase_period`, was created by grouping purchase hours into four time periods: Night, Morning, Afternoon, and Evening.

This feature provides a more interpretable representation of customer purchasing behavior and supports analyses of shopping patterns across different periods of the day.

In [ ]:
def get_purchase_period(hour):
    if 0 <= hour < 6:
        return "Night"
    elif 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 18:
        return "Afternoon"
    else:
        return "Evening"

orders["purchase_period"] = orders["purchase_hour"].apply(get_purchase_period)
orders["purchase_period"].value_counts()

purchase_period
Afternoon    38361
Evening      34100
Morning      22240
Night         4740
Name: count, dtype: int64

ORDERS FEATURE ENGINEERING BİTTİ